# Carga de Datos

In [1]:
import pandas as pd

# Cargamos la base de datos de préstamos
ruta = "data/loan.csv"  # descargar desde Kaggle (ver README)
df = pd.read_csv(ruta, low_memory=False)

# ¿Cuántas filas y columnas tiene?
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 39717
Columnas: 111


## Solo columnas que se van a usar en el Análisis

In [3]:
# Elegimos solo las columnas útiles para el análisis de riesgo
columnas = [
    "id",                    # identificador del préstamo
    "issue_d",               # fecha de desembolso (para cosechas)
    "loan_status",           # estado del crédito (bueno/malo)
    "loan_amnt",             # monto solicitado
    "funded_amnt",           # monto desembolsado
    "term",                  # plazo (36 o 60 meses)
    "int_rate",              # tasa de interés
    "installment",           # cuota mensual
    "grade",                 # calificación de riesgo (A-G)
    "sub_grade",             # subcalificación
    "emp_length",            # antigüedad laboral
    "home_ownership",        # situación de vivienda
    "annual_inc",            # ingreso anual
    "dti",                   # ratio deuda/ingreso
    "purpose",               # motivo del préstamo
    "addr_state",            # estado/región
]

# Creamos una nueva tabla solo con esas columnas
df = df[columnas]

print("Ahora la base tiene:", df.shape[0], "filas y", df.shape[1], "columnas")
df.head()

Ahora la base tiene: 39717 filas y 16 columnas


,id,issue_d,loan_status,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,dti,purpose,addr_state
0,1077501,Dec-11,Fully Paid,5000,5000,36 months,10.65%,162.87,B,B2,10+ years,RENT,24000.0,27.65,credit_card,AZ
1,1077430,Dec-11,Charged Off,2500,2500,60 months,15.27%,59.83,C,C4,< 1 year,RENT,30000.0,1.00,car,GA
2,1077175,Dec-11,Fully Paid,2400,2400,36 months,15.96%,84.33,C,C5,10+ years,RENT,12252.0,8.72,small_business,IL
3,1076863,Dec-11,Fully Paid,10000,10000,36 months,13.49%,339.31,C,C1,10+ years,RENT,49200.0,20.00,other,CA
4,1075358,Dec-11,Current,3000,3000,60 months,12.69%,67.79,B,B5,1 year,RENT,80000.0,17.94,other,OR


## Arreglar los formatos de columnas:

In [4]:
# Primero hacemos una copia, para trabajar tranquilas y sin advertencias
df = df.copy()

# 1) int_rate: de texto "10.65%" a número 10.65
df["int_rate"] = df["int_rate"].str.replace("%", "", regex=False).astype(float)

# 2) term: de texto " 36 months" a número entero 36
df["term"] = df["term"].str.replace("months", "", regex=False).str.strip().astype(int)

# 3) issue_d: de texto "Dec-11" a fecha real
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%y")

# Verificamos que los tipos de dato cambiaron
print(df[["int_rate", "term", "issue_d"]].dtypes)
print()
df[["int_rate", "term", "issue_d"]].head()

int_rate           float64
term                 int64
issue_d     datetime64[us]
dtype: object



,int_rate,term,issue_d
0,10.65,36,2011-12-01
1,15.27,60,2011-12-01
2,15.96,36,2011-12-01
3,13.49,36,2011-12-01
4,12.69,60,2011-12-01


## Crear Metrica de Interes

In [5]:
# 1) Traducimos el estado a español, más claro para el dashboard
mapa_estado = {
    "Fully Paid":  "Pagado",
    "Charged Off": "Incumplido",
    "Current":     "Vigente",
}
df["estado"] = df["loan_status"].map(mapa_estado)

# 2) Bandera de incumplimiento: 1 = incumplió, 0 = pagó, vacío = vigente (aún no sabemos su final)
df["incumplio"] = df["loan_status"].map({"Charged Off": 1, "Fully Paid": 0})

# 3) Revisamos cómo quedó
print(df["estado"].value_counts())
print()
print("Tasa de incumplimiento (solo créditos ya cerrados):",
      round(df["incumplio"].mean() * 100, 2), "%")

estado
Pagado        32950
Incumplido     5627
Vigente        1140
Name: count, dtype: int64

Tasa de incumplimiento (solo créditos ya cerrados): 14.59 %


## Revisar los datos faltantes

In [6]:
# ¿Cuántos vacíos quedan en cada columna?
df.isna().sum()

id                   0
issue_d              0
loan_status          0
loan_amnt            0
funded_amnt          0
term                 0
int_rate             0
installment          0
grade                0
sub_grade            0
emp_length        1075
home_ownership       0
annual_inc           0
dti                  0
purpose              0
addr_state           0
estado               0
incumplio         1140
dtype: int64

## Etiquetar el último vacío y GUARDAR tu base limpia

In [6]:
# 1) emp_length: rellenamos los vacíos con una etiqueta clara
df["emp_length"] = df["emp_length"].fillna("No informado")

# 2) Guardamos la base limpia en la carpeta del proyecto
ruta_limpia = "data/loan_limpio.csv"
df.to_csv(ruta_limpia, index=False)

print("¡Guardado! Base limpia:", df.shape[0], "filas y", df.shape[1], "columnas")

¡Guardado! Base limpia: 39717 filas y 18 columnas
